# JAX Addition Transformer: exact 10M

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marcoharuni/jax-addition-transformer/blob/main/notebooks/01_build_train_exact_10m.ipynb)

A complete decoder-only arithmetic-language-model experiment. Numerical outputs are produced only when this notebook runs.

## 1. Runtime and device verification

In [ ]:
import sys, subprocess, importlib.metadata
print(sys.version)
import jax
print('JAX',jax.__version__,'devices',jax.devices())
FULL_RUN=any(d.platform=='gpu' for d in jax.devices())
if not FULL_RUN: print('The full default run requires Runtime > Change runtime type > T4 GPU.')
def require_gpu_for_full_run():
    assert any(d.platform=='gpu' for d in jax.devices()), 'Select a T4 GPU runtime before starting the full default run.'
support={'flax':'0.12.0','optax':'0.2.6','orbax-checkpoint':'0.11.28','matplotlib':'3.10.7'}
needed=[]
for package,version in support.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version: needed.append(f'{package}=={version}')
if needed: subprocess.check_call([sys.executable,'-m','pip','install',*needed])
for package in ['jax','jaxlib',*support]: print(package,importlib.metadata.version(package))

## 2. Reproducibility and task representation

`AAA + BBB = RRRR` uses a reversed four-digit answer so carries flow in generation order. The 15-token input predicts the shifted sequence, with loss only at target positions 11–14.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 2)); ax.axis('off')
for i, token in enumerate('123 + 456 = 9750'):
    ax.add_patch(plt.Rectangle((i,0), .9,.8, fill=False)); ax.text(i+.45,.4,repr(token)[1:-1] or 'space',ha='center',va='center')
ax.set_xlim(0,16); ax.set_ylim(0,1); ax.set_title('Fixed-width token layout'); plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,2)); ax.axis('off'); ax.text(.1,.7,'normal answer: 0579',fontsize=16); ax.annotate('',(.75,.4),(.4,.4),arrowprops={'arrowstyle':'->'}); ax.text(.1,.15,'generation: 9750',fontsize=16); ax.set_title('Reverse the answer for least-significant-digit-first generation'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,2)); ax.axis('off')
labels=['token + position','5 decoder blocks','final LayerNorm','tied token head']
for i,label in enumerate(labels):
 ax.add_patch(plt.Rectangle((i*2.4,.2),2,.6,fill=False)); ax.text(i*2.4+1,.5,label,ha='center',va='center')
 if i<len(labels)-1: ax.annotate('',(i*2.4+2.35,.5),(i*2.4+2,.5),arrowprops={'arrowstyle':'->'})
ax.set_xlim(0,9.5); ax.set_ylim(0,1); ax.set_title('Decoder-only transformer architecture'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11,2)); ax.axis('off')
labels=['x','LayerNorm','causal MHA','residual +','LayerNorm','GELU FFN','residual +']
for i,label in enumerate(labels): ax.text(i/7+.06,.5,label,ha='center',bbox={'fill':False,'boxstyle':'round'})
for i in range(6): ax.annotate('',((i+1)/7+.01,.5),(i/7+.11,.5),arrowprops={'arrowstyle':'->'})
ax.set_title('One Pre-LayerNorm block data flow'); plt.show()

In [ ]:
shape_rows=[['input','B × 15 × 320'],['Q, K, V','B × 15 × 5 × 64'],['scores','B × 5 × 15 × 15'],['attention output','B × 15 × 5 × 64'],['projection','B × 15 × 320']]
fig,ax=plt.subplots(figsize=(7,2.5)); ax.axis('off'); ax.table(cellText=shape_rows,colLabels=['tensor','shape'],loc='center'); ax.set_title('MHA tensor-shape walkthrough'); plt.show()

## 3. Canonical implementation (visible and written locally)

In [ ]:
from pathlib import Path
Path('src/jax_addition_transformer').mkdir(parents=True, exist_ok=True)

### Canonical source: `config.py`

In [ ]:
%%writefile src/jax_addition_transformer/config.py
"""Validated experiment configuration."""

from __future__ import annotations

import dataclasses
import hashlib
import json
from pathlib import Path
from typing import Any


@dataclasses.dataclass(frozen=True)
class TaskConfig:
    max_digits: int = 3

    def __post_init__(self) -> None:
        if self.max_digits < 1:
            raise ValueError("max_digits must be positive")

    @property
    def answer_digits(self) -> int:
        return self.max_digits + 1

    @property
    def full_sequence_length(self) -> int:
        return 3 * self.max_digits + 7

    @property
    def model_input_length(self) -> int:
        return self.full_sequence_length - 1

    @property
    def prompt_length(self) -> int:
        return 2 * self.max_digits + 6

    @property
    def maximum_operand(self) -> int:
        return 10**self.max_digits - 1


@dataclasses.dataclass(frozen=True)
class ModelConfig:
    n_layers: int = 5
    d_model: int = 320
    n_heads: int = 5
    n_kv_heads: int = 5
    d_ff: int = 2480
    max_input_length: int = 15
    vocab_size: int = 13
    norm_type: str = "layernorm"
    position_type: str = "learned"
    ffn_type: str = "gelu"
    attention_type: str = "mha"
    use_bias: bool = False
    tie_embeddings: bool = True
    compute_dtype: str = "float16"
    parameter_dtype: str = "float32"
    norm_epsilon: float = 1e-5
    rope_base: float = 10000.0
    init_scale: float = 0.02

    def __post_init__(self) -> None:
        choices = {
            "norm_type": (self.norm_type, {"layernorm", "rmsnorm"}),
            "position_type": (self.position_type, {"learned", "rope"}),
            "ffn_type": (self.ffn_type, {"gelu", "relu", "relu2", "swiglu", "geglu"}),
            "attention_type": (self.attention_type, {"mha", "gqa", "mqa"}),
        }
        for name, (value, allowed) in choices.items():
            if value not in allowed:
                raise ValueError(f"{name} must be one of {sorted(allowed)}")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        if self.n_heads % self.n_kv_heads:
            raise ValueError("n_heads must be divisible by n_kv_heads")
        expected_kv = {"mha": self.n_heads, "mqa": 1}.get(self.attention_type)
        if expected_kv is not None and self.n_kv_heads != expected_kv:
            raise ValueError(f"{self.attention_type} requires n_kv_heads={expected_kv}")
        if min(self.n_layers, self.d_model, self.n_heads, self.n_kv_heads, self.d_ff) < 1:
            raise ValueError("model dimensions must be positive")

    @property
    def head_dim(self) -> int:
        return self.d_model // self.n_heads

    @property
    def is_exact_default(self) -> bool:
        return self == ModelConfig()


@dataclasses.dataclass(frozen=True)
class OptimizerConfig:
    name: str = "adamw"
    peak_learning_rate: float = 1e-3
    initial_learning_rate: float = 0.0
    final_learning_rate: float = 1e-4
    warmup_steps: int = 100
    total_steps: int = 5000
    beta1: float = 0.9
    beta2: float = 0.99
    epsilon: float = 1e-8
    weight_decay: float = 0.1
    clip_global_norm: float = 1.0
    momentum: float = 0.9

    def __post_init__(self) -> None:
        if self.name not in {"adamw", "adam", "sgd"}:
            raise ValueError("optimizer name must be adamw, adam, or sgd")
        if not 0 <= self.warmup_steps < self.total_steps:
            raise ValueError("warmup_steps must be in [0, total_steps)")


@dataclasses.dataclass(frozen=True)
class TrainingConfig:
    seed: int = 42
    train_pairs: int = 200_000
    validation_pairs: int = 20_000
    test_pairs: int = 780_000
    batch_size: int = 2048
    evaluation_batch_size: int = 4000
    max_steps: int = 5000
    validation_interval: int = 250
    logging_interval: int = 25
    checkpoint_interval: int = 500


@dataclasses.dataclass(frozen=True)
class ExperimentConfig:
    model: ModelConfig = dataclasses.field(default_factory=ModelConfig)
    task: TaskConfig = dataclasses.field(default_factory=TaskConfig)
    optimizer: OptimizerConfig = dataclasses.field(default_factory=OptimizerConfig)
    training: TrainingConfig = dataclasses.field(default_factory=TrainingConfig)

    def __post_init__(self) -> None:
        if self.model.max_input_length != self.task.model_input_length:
            raise ValueError("model.max_input_length must match task.model_input_length")
        domain = (self.task.maximum_operand + 1) ** 2
        if (
            self.training.train_pairs + self.training.validation_pairs + self.training.test_pairs
            != domain
        ):
            raise ValueError("split sizes must sum to the complete ordered-pair domain")

    def as_dict(self) -> dict[str, Any]:
        return dataclasses.asdict(self)

    @property
    def fingerprint(self) -> str:
        raw = json.dumps(self.as_dict(), sort_keys=True, separators=(",", ":")).encode()
        return hashlib.sha256(raw).hexdigest()

    @classmethod
    def from_dict(cls, raw: dict[str, Any]) -> ExperimentConfig:
        return cls(
            ModelConfig(**raw["model"]),
            TaskConfig(**raw["task"]),
            OptimizerConfig(**raw["optimizer"]),
            TrainingConfig(**raw["training"]),
        )

    @classmethod
    def load(cls, path: str | Path) -> ExperimentConfig:
        return cls.from_dict(json.loads(Path(path).read_text()))

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.as_dict(), indent=2) + "\n")


### Canonical source: `tokenizer.py`

In [ ]:
%%writefile src/jax_addition_transformer/tokenizer.py
"""Stable 13-character vocabulary and arithmetic formatting."""

from __future__ import annotations

import re

import numpy as np

TOKENS = "0123456789 +="
TOKEN_TO_ID = {token: index for index, token in enumerate(TOKENS)}
ID_TO_TOKEN = dict(enumerate(TOKENS))


def encode(text: str) -> np.ndarray:
    try:
        return np.asarray([TOKEN_TO_ID[c] for c in text], dtype=np.int32)
    except KeyError as exc:
        raise ValueError(f"character {exc.args[0]!r} is outside the 13-token vocabulary") from None


def decode(ids: np.ndarray | list[int]) -> str:
    values = np.asarray(ids)
    if values.ndim != 1:
        raise ValueError("token IDs must be one-dimensional")
    if np.any((values < 0) | (values >= len(TOKENS))):
        raise ValueError("token ID outside [0, 12]")
    return "".join(ID_TO_TOKEN[int(i)] for i in values)


def validate_operand(value: int, max_digits: int = 3) -> int:
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)):
        raise TypeError("operands must be integers")
    value = int(value)
    if not 0 <= value < 10**max_digits:
        raise ValueError(f"operand must be between 0 and {10**max_digits - 1}")
    return value


def format_prompt(a: int, b: int, max_digits: int = 3) -> str:
    a, b = validate_operand(a, max_digits), validate_operand(b, max_digits)
    return f"{a:0{max_digits}d} + {b:0{max_digits}d} = "


def format_training_example(a: int, b: int, max_digits: int = 3) -> str:
    prompt = format_prompt(a, b, max_digits)
    answer = f"{a + b:0{max_digits + 1}d}"[::-1]
    return prompt + answer


def decode_internal_answer(ids: np.ndarray | list[int]) -> str:
    internal = decode(ids)
    if not internal or any(c not in "0123456789" for c in internal):
        raise ValueError("generated answer must contain digits only")
    return internal[::-1].lstrip("0") or "0"


_EXPRESSION = re.compile(r"^\s*(\d+)\s*\+\s*(\d+)\s*(?:=\s*)?$")


def parse_expression(expression: str, max_digits: int = 3) -> tuple[int, int]:
    if not isinstance(expression, str) or (match := _EXPRESSION.fullmatch(expression)) is None:
        raise ValueError("expected an expression such as '123 + 456' or '123+456='")
    return validate_operand(int(match[1]), max_digits), validate_operand(int(match[2]), max_digits)


### Canonical source: `data.py`

In [ ]:
%%writefile src/jax_addition_transformer/data.py
"""Vectorized complete-domain data, carries, and deterministic stratified splits."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass

import numpy as np


def pair_ids_to_operands(pair_ids: np.ndarray, base: int = 1000) -> tuple[np.ndarray, np.ndarray]:
    ids = np.asarray(pair_ids, dtype=np.int64)
    if np.any((ids < 0) | (ids >= base * base)):
        raise ValueError("pair ID outside domain")
    return ids // base, ids % base


def operands_to_pair_ids(a: np.ndarray, b: np.ndarray, base: int = 1000) -> np.ndarray:
    a, b = np.asarray(a), np.asarray(b)
    if np.any((a < 0) | (a >= base) | (b < 0) | (b >= base)):
        raise ValueError("operand outside domain")
    return (base * a + b).astype(np.int64)


def operand_lengths(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    if np.any(values < 0):
        raise ValueError("operand lengths require non-negative integers")
    boundaries = 10 ** np.arange(1, 19, dtype=np.int64)
    return (np.searchsorted(boundaries, values, side="right") + 1).astype(np.int8)


def carry_bits(a: np.ndarray | int, b: np.ndarray | int, max_digits: int = 3) -> np.ndarray:
    """Return units-to-most-significant carry bits; column zero is units."""
    a, b = np.asarray(a), np.asarray(b)
    carry = np.zeros(np.broadcast_shapes(a.shape, b.shape), dtype=np.int8)
    result = []
    for power in range(max_digits):
        total = (a // (10**power)) % 10 + (b // (10**power)) % 10 + carry
        carry = (total >= 10).astype(np.int8)
        result.append(carry)
    return np.stack(result, axis=-1)


def carry_pattern(
    a: np.ndarray | int, b: np.ndarray | int, max_digits: int = 3
) -> np.ndarray | str:
    bits = carry_bits(a, b, max_digits)
    if bits.ndim == 1:
        return "".join(str(int(x)) for x in bits)
    weights = 2 ** np.arange(max_digits - 1, -1, -1)
    return (bits * weights).sum(axis=-1).astype(np.int8)


def stratum_codes(a: np.ndarray, b: np.ndarray, max_digits: int = 3) -> np.ndarray:
    carry_states = 2**max_digits
    return (
        ((operand_lengths(a) - 1) * max_digits + (operand_lengths(b) - 1)) * carry_states
        + carry_pattern(a, b, max_digits)
    ).astype(np.int16)


def tokenize_pairs(pair_ids: np.ndarray, max_digits: int = 3) -> np.ndarray:
    """Vectorize AAA + BBB = RRRR without allocating Python strings."""
    base = 10**max_digits
    a, b = pair_ids_to_operands(pair_ids, base)
    answer = a + b
    n = len(a)
    full = np.empty((n, 3 * max_digits + 7), dtype=np.int32)
    powers = 10 ** np.arange(max_digits - 1, -1, -1)
    full[:, :max_digits] = (a[:, None] // powers) % 10
    full[:, max_digits : max_digits + 3] = (10, 11, 10)
    start = max_digits + 3
    full[:, start : start + max_digits] = (b[:, None] // powers) % 10
    full[:, start + max_digits : start + max_digits + 3] = (10, 12, 10)
    answer_powers = 10 ** np.arange(0, max_digits + 1)
    full[:, -max_digits - 1 :] = (answer[:, None] // answer_powers) % 10
    return full


@dataclass(frozen=True)
class DatasetSplit:
    train: np.ndarray
    validation: np.ndarray
    test: np.ndarray
    seed: int
    fingerprint: str


def _largest_remainder(counts: np.ndarray, total: int, capacities: np.ndarray) -> np.ndarray:
    ideal = counts.astype(np.float64) * total / counts.sum()
    allocated = np.minimum(np.floor(ideal).astype(np.int64), capacities)
    order = np.lexsort((np.arange(len(counts)), -(ideal - allocated)))
    remaining = total - int(allocated.sum())
    while remaining:
        eligible = order[allocated[order] < capacities[order]]
        take = eligible[:remaining]
        allocated[take] += 1
        remaining -= len(take)
    return allocated


def create_split(
    seed: int = 42, train_size: int = 200_000, validation_size: int = 20_000, base: int = 1000
) -> DatasetSplit:
    ids = np.arange(base * base, dtype=np.int64)
    a, b = pair_ids_to_operands(ids, base)
    max_digits = len(str(base - 1))
    codes = stratum_codes(a, b, max_digits)
    unique, counts = np.unique(codes, return_counts=True)
    train_counts = _largest_remainder(counts, train_size, counts)
    validation_counts = _largest_remainder(counts, validation_size, counts - train_counts)
    rng = np.random.default_rng(seed)
    train_parts, validation_parts, test_parts = [], [], []
    for code, n_train, n_validation in zip(unique, train_counts, validation_counts, strict=True):
        members = ids[codes == code].copy()
        rng.shuffle(members)
        train_parts.append(members[:n_train])
        validation_parts.append(members[n_train : n_train + n_validation])
        test_parts.append(members[n_train + n_validation :])
    train = np.concatenate(train_parts)
    validation = np.concatenate(validation_parts)
    test = np.concatenate(test_parts)
    rng.shuffle(train)
    rng.shuffle(validation)
    rng.shuffle(test)
    digest = hashlib.sha256()
    for name, values in (("train", train), ("validation", validation), ("test", test)):
        digest.update(name.encode())
        digest.update(values.astype("<i8", copy=False).tobytes())
    return DatasetSplit(train, validation, test, seed, digest.hexdigest())


### Canonical source: `sampling.py`

In [ ]:
%%writefile src/jax_addition_transformer/sampling.py
"""Explicit-state hybrid training sampler."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .data import pair_ids_to_operands, stratum_codes, tokenize_pairs


@dataclass
class HybridSampler:
    train_ids: np.ndarray
    batch_size: int
    rng: np.random.Generator
    base: int = 1000

    def __post_init__(self) -> None:
        self.train_ids = np.asarray(self.train_ids, dtype=np.int64)
        a, b = pair_ids_to_operands(self.train_ids, self.base)
        codes = stratum_codes(a, b, len(str(self.base - 1)))
        self.strata = [self.train_ids[codes == code] for code in np.unique(codes)]
        if self.batch_size < 2:
            raise ValueError("batch_size must be at least two")

    @classmethod
    def create(
        cls, train_ids: np.ndarray, batch_size: int, seed: int, base: int = 1000
    ) -> HybridSampler:
        return cls(train_ids, batch_size, np.random.default_rng(seed), base)

    @property
    def state(self) -> dict:
        return self.rng.bit_generator.state

    def restore_state(self, state: dict) -> None:
        self.rng.bit_generator.state = state

    def sample_ids(self) -> np.ndarray:
        natural_n = self.batch_size // 2
        balanced_n = self.batch_size - natural_n
        natural = self.rng.choice(self.train_ids, natural_n, replace=True)
        chosen_strata = self.rng.integers(0, len(self.strata), balanced_n)
        balanced = np.fromiter(
            (self.rng.choice(self.strata[i]) for i in chosen_strata),
            dtype=np.int64,
            count=balanced_n,
        )
        result = np.concatenate((natural, balanced))
        self.rng.shuffle(result)
        return result

    def sample_batch(self, max_digits: int = 3) -> tuple[np.ndarray, np.ndarray]:
        sequences = tokenize_pairs(self.sample_ids(), max_digits)
        return sequences[:, :-1], sequences[:, 1:]


### Canonical source: `layers.py`

In [ ]:
%%writefile src/jax_addition_transformer/layers.py
"""Handwritten parameter initialization, embedding lookup, and projection."""

from __future__ import annotations

import jax
import jax.numpy as jnp
from flax import nnx


def dtype_from_name(name: str) -> jnp.dtype:
    try:
        return {"float16": jnp.float16, "float32": jnp.float32, "bfloat16": jnp.bfloat16}[name]
    except KeyError:
        raise ValueError(f"unsupported dtype {name!r}") from None


def normal_parameter(
    rngs: nnx.Rngs, shape: tuple[int, ...], scale: float, dtype: jnp.dtype
) -> nnx.Param:
    """Initialize W_ij ~ Normal(0, scale^2), explicitly."""
    return nnx.Param(
        (jax.random.normal(rngs.params(), shape, dtype=jnp.float32) * scale).astype(dtype)
    )


class Linear(nnx.Module):
    def __init__(
        self,
        inputs: int,
        outputs: int,
        *,
        rngs: nnx.Rngs,
        scale: float = 0.02,
        use_bias: bool = False,
        parameter_dtype: jnp.dtype = jnp.float32,
    ):
        self.kernel = normal_parameter(rngs, (inputs, outputs), scale, parameter_dtype)
        self.bias = nnx.Param(jnp.zeros((outputs,), parameter_dtype)) if use_bias else None

    def __call__(self, x: jax.Array, compute_dtype: jnp.dtype = jnp.float32) -> jax.Array:
        lhs, rhs = x.astype(compute_dtype), self.kernel.value.astype(compute_dtype)
        y = jax.lax.dot_general(
            lhs, rhs, (((lhs.ndim - 1,), (0,)), ((), ())), preferred_element_type=jnp.float32
        )
        return y + self.bias.value.astype(jnp.float32) if self.bias is not None else y


def embedding_lookup(table: jax.Array, token_ids: jax.Array) -> jax.Array:
    if not jnp.issubdtype(token_ids.dtype, jnp.integer):
        raise TypeError("token IDs must be integers")
    return jnp.take(table, token_ids, axis=0)


### Canonical source: `normalization.py`

In [ ]:
%%writefile src/jax_addition_transformer/normalization.py
"""Handwritten LayerNorm and RMSNorm."""

from __future__ import annotations

import jax
import jax.numpy as jnp
from flax import nnx


class LayerNorm(nnx.Module):
    def __init__(self, features: int, epsilon: float = 1e-5, dtype: jnp.dtype = jnp.float32):
        self.scale = nnx.Param(jnp.ones((features,), dtype))
        self.bias = nnx.Param(jnp.zeros((features,), dtype))
        self.epsilon = epsilon

    def __call__(self, x: jax.Array) -> jax.Array:
        x32 = x.astype(jnp.float32)
        mean = jnp.mean(x32, axis=-1, keepdims=True)
        variance = jnp.mean(jnp.square(x32 - mean), axis=-1, keepdims=True)
        return (x32 - mean) * jax.lax.rsqrt(
            variance + self.epsilon
        ) * self.scale.value + self.bias.value


class RMSNorm(nnx.Module):
    def __init__(self, features: int, epsilon: float = 1e-5, dtype: jnp.dtype = jnp.float32):
        self.scale = nnx.Param(jnp.ones((features,), dtype))
        self.epsilon = epsilon

    def __call__(self, x: jax.Array) -> jax.Array:
        x32 = x.astype(jnp.float32)
        return (
            x32
            * jax.lax.rsqrt(jnp.mean(jnp.square(x32), axis=-1, keepdims=True) + self.epsilon)
            * self.scale.value
        )


def make_norm(kind: str, features: int, epsilon: float, dtype: jnp.dtype) -> nnx.Module:
    return (
        LayerNorm(features, epsilon, dtype)
        if kind == "layernorm"
        else RMSNorm(features, epsilon, dtype)
    )


### Canonical source: `positional.py`

In [ ]:
%%writefile src/jax_addition_transformer/positional.py
"""Learned absolute positions and rotary position encoding."""

from __future__ import annotations

import jax
import jax.numpy as jnp
from flax import nnx

from .layers import normal_parameter


class LearnedPositions(nnx.Module):
    def __init__(self, length: int, width: int, *, rngs: nnx.Rngs, scale: float, dtype: jnp.dtype):
        self.embedding = normal_parameter(rngs, (length, width), scale, dtype)

    def __call__(self, length: int) -> jax.Array:
        return self.embedding.value[:length]


def apply_rope(x: jax.Array, base: float = 10_000.0) -> jax.Array:
    """Rotate adjacent feature pairs; x is [B, T, H, D]."""
    width = x.shape[-1]
    if width % 2:
        raise ValueError("RoPE head dimension must be even")
    positions = jnp.arange(x.shape[1], dtype=jnp.float32)
    frequencies = base ** (-jnp.arange(0, width, 2, dtype=jnp.float32) / width)
    angles = positions[:, None] * frequencies[None, :]
    cos, sin = jnp.cos(angles)[None, :, None, :], jnp.sin(angles)[None, :, None, :]
    even, odd = x[..., 0::2].astype(jnp.float32), x[..., 1::2].astype(jnp.float32)
    return jnp.stack((even * cos - odd * sin, even * sin + odd * cos), axis=-1).reshape(x.shape)


### Canonical source: `attention.py`

In [ ]:
%%writefile src/jax_addition_transformer/attention.py
"""Handwritten MHA, GQA, MQA, causal mask, and float32 attention."""

from __future__ import annotations

import math

import jax
import jax.numpy as jnp
from flax import nnx

from .config import ModelConfig
from .layers import Linear, dtype_from_name
from .positional import apply_rope


def causal_mask(length: int) -> jax.Array:
    return jnp.arange(length)[:, None] >= jnp.arange(length)[None, :]


def scaled_dot_product_attention(
    q: jax.Array, k: jax.Array, v: jax.Array
) -> tuple[jax.Array, jax.Array]:
    """Attend with q [B,T,H,D] and k/v [B,T,K,D], repeating KV groups logically."""
    repeats = q.shape[2] // k.shape[2]
    k = jnp.repeat(k, repeats, axis=2)
    v = jnp.repeat(v, repeats, axis=2)
    scores = jnp.einsum(
        "bthd,bshd->bhts",
        q.astype(jnp.float32),
        k.astype(jnp.float32),
        preferred_element_type=jnp.float32,
    )
    scores = scores / math.sqrt(q.shape[-1])
    visible = causal_mask(q.shape[1])[None, None, :, :]
    scores = jnp.where(visible, scores, jnp.finfo(jnp.float32).min)
    probabilities = jax.nn.softmax(scores, axis=-1).astype(jnp.float32)
    output = jnp.einsum(
        "bhts,bshd->bthd", probabilities, v.astype(jnp.float32), preferred_element_type=jnp.float32
    )
    return output, probabilities


class CausalAttention(nnx.Module):
    def __init__(self, config: ModelConfig, *, rngs: nnx.Rngs):
        d, kv = config.d_model, config.n_kv_heads * config.head_dim
        dtype = dtype_from_name(config.parameter_dtype)
        args = {
            "rngs": rngs,
            "scale": config.init_scale,
            "use_bias": config.use_bias,
            "parameter_dtype": dtype,
        }
        self.q_proj = Linear(d, d, **args)
        self.k_proj = Linear(d, kv, **args)
        self.v_proj = Linear(d, kv, **args)
        self.out_proj = Linear(d, d, **args)
        self.config = config

    def __call__(
        self, x: jax.Array, return_attention: bool = False
    ) -> jax.Array | tuple[jax.Array, jax.Array]:
        cfg, batch, length = self.config, x.shape[0], x.shape[1]
        compute = dtype_from_name(cfg.compute_dtype)
        q = self.q_proj(x, compute).reshape(batch, length, cfg.n_heads, cfg.head_dim)
        k = self.k_proj(x, compute).reshape(batch, length, cfg.n_kv_heads, cfg.head_dim)
        v = self.v_proj(x, compute).reshape(batch, length, cfg.n_kv_heads, cfg.head_dim)
        if cfg.position_type == "rope":
            q, k = apply_rope(q, cfg.rope_base), apply_rope(k, cfg.rope_base)
        attended, probabilities = scaled_dot_product_attention(q, k, v)
        output = self.out_proj(attended.reshape(batch, length, cfg.d_model), compute)
        return (output, probabilities) if return_attention else output


### Canonical source: `ffn.py`

In [ ]:
%%writefile src/jax_addition_transformer/ffn.py
"""Handwritten GELU, ReLU, squared ReLU, SiLU, SwiGLU, and GEGLU FFNs."""

from __future__ import annotations

import jax
import jax.numpy as jnp
from flax import nnx

from .config import ModelConfig
from .layers import Linear, dtype_from_name


def gelu(x: jax.Array) -> jax.Array:
    return 0.5 * x * (1.0 + jax.lax.erf(x / jnp.sqrt(2.0)))


def silu(x: jax.Array) -> jax.Array:
    return x * jax.nn.sigmoid(x)


class FeedForward(nnx.Module):
    def __init__(self, config: ModelConfig, *, rngs: nnx.Rngs):
        dtype = dtype_from_name(config.parameter_dtype)
        args = {
            "rngs": rngs,
            "scale": config.init_scale,
            "use_bias": config.use_bias,
            "parameter_dtype": dtype,
        }
        self.in_proj = Linear(config.d_model, config.d_ff, **args)
        self.gate_proj = (
            Linear(config.d_model, config.d_ff, **args)
            if config.ffn_type in {"swiglu", "geglu"}
            else None
        )
        self.out_proj = Linear(config.d_ff, config.d_model, **args)
        self.config = config

    def __call__(self, x: jax.Array) -> jax.Array:
        compute = dtype_from_name(self.config.compute_dtype)
        hidden = self.in_proj(x, compute)
        kind = self.config.ffn_type
        if kind == "gelu":
            activated = gelu(hidden)
        elif kind == "relu":
            activated = jax.nn.relu(hidden)
        elif kind == "relu2":
            activated = jnp.square(jax.nn.relu(hidden))
        elif kind == "swiglu":
            activated = silu(hidden) * self.gate_proj(x, compute)
        else:
            activated = gelu(hidden) * self.gate_proj(x, compute)
        return self.out_proj(activated, compute)


### Canonical source: `model.py`

In [ ]:
%%writefile src/jax_addition_transformer/model.py
"""Decoder-only Pre-LayerNorm transformer with directly tied output weights."""

from __future__ import annotations


import jax
import jax.numpy as jnp
from flax import nnx

from .attention import CausalAttention
from .config import ModelConfig
from .ffn import FeedForward
from .layers import dtype_from_name, embedding_lookup, normal_parameter
from .normalization import make_norm
from .positional import LearnedPositions


class TransformerBlock(nnx.Module):
    def __init__(self, config: ModelConfig, *, rngs: nnx.Rngs):
        dtype = dtype_from_name(config.parameter_dtype)
        self.attention_norm = make_norm(
            config.norm_type, config.d_model, config.norm_epsilon, dtype
        )
        self.attention = CausalAttention(config, rngs=rngs)
        self.ffn_norm = make_norm(config.norm_type, config.d_model, config.norm_epsilon, dtype)
        self.ffn = FeedForward(config, rngs=rngs)

    def __call__(self, x: jax.Array, return_attention: bool = False):
        result = self.attention(self.attention_norm(x), return_attention)
        if return_attention:
            attended, probabilities = result
        else:
            attended = result
        x = x + attended
        x = x + self.ffn(self.ffn_norm(x))
        return (x, probabilities) if return_attention else x


class AdditionTransformer(nnx.Module):
    def __init__(self, config: ModelConfig, *, rngs: nnx.Rngs):
        dtype = dtype_from_name(config.parameter_dtype)
        self.token_embedding = normal_parameter(
            rngs, (config.vocab_size, config.d_model), config.init_scale, dtype
        )
        self.positions = (
            LearnedPositions(
                config.max_input_length,
                config.d_model,
                rngs=rngs,
                scale=config.init_scale,
                dtype=dtype,
            )
            if config.position_type == "learned"
            else None
        )
        self.blocks = nnx.List(
            [TransformerBlock(config, rngs=rngs) for _ in range(config.n_layers)]
        )
        self.final_norm = make_norm(config.norm_type, config.d_model, config.norm_epsilon, dtype)
        self.lm_head = (
            None
            if config.tie_embeddings
            else normal_parameter(
                rngs, (config.d_model, config.vocab_size), config.init_scale, dtype
            )
        )
        self.config = config

    def __call__(self, token_ids: jax.Array, return_attention: bool = False):
        if token_ids.ndim != 2 or token_ids.shape[1] > self.config.max_input_length:
            raise ValueError("token_ids must be [batch, sequence<=max_input_length]")
        x = embedding_lookup(self.token_embedding.value, token_ids)
        if self.positions is not None:
            x = x + self.positions(token_ids.shape[1])
        maps = []
        for block in self.blocks:
            if return_attention:
                x, probabilities = block(x, True)
                maps.append(probabilities)
            else:
                x = block(x)
        x = self.final_norm(x)
        output_weight = self.token_embedding.value.T if self.lm_head is None else self.lm_head.value
        logits = jnp.einsum(
            "btd,dv->btv",
            x.astype(jnp.float32),
            output_weight.astype(jnp.float32),
            preferred_element_type=jnp.float32,
        )
        return (logits, maps) if return_attention else logits


def count_parameters(model: AdditionTransformer) -> int:
    state = nnx.state(model, nnx.Param)
    return sum(int(x.size) for x in jax.tree.leaves(state))


def parameter_table(config: ModelConfig) -> list[tuple[str, int]]:
    bias = 0
    if config.use_bias:
        bias = (
            config.d_model
            + 2 * config.n_kv_heads * config.head_dim
            + config.d_model
            + config.d_ff
            + config.d_model
        )
        if config.ffn_type in {"swiglu", "geglu"}:
            bias += config.d_ff
    attention = (
        config.d_model * config.d_model * 2
        + 2 * config.d_model * config.n_kv_heads * config.head_dim
    )
    ffn = 2 * config.d_model * config.d_ff + (
        config.d_model * config.d_ff if config.ffn_type in {"swiglu", "geglu"} else 0
    )
    norms = 4 * config.d_model if config.norm_type == "layernorm" else 2 * config.d_model
    rows = [
        ("attention projections", config.n_layers * attention),
        ("feed-forward networks", config.n_layers * ffn),
        ("block normalizations", config.n_layers * norms),
    ]
    if bias:
        rows.append(("linear biases", config.n_layers * bias))
    rows.append(("token embedding", config.vocab_size * config.d_model))
    rows.append(
        (
            "position embedding",
            config.max_input_length * config.d_model if config.position_type == "learned" else 0,
        )
    )
    rows.append(
        ("final norm", 2 * config.d_model if config.norm_type == "layernorm" else config.d_model)
    )
    if not config.tie_embeddings:
        rows.append(("language-model head", config.d_model * config.vocab_size))
    return rows


def assert_parameter_count(model: AdditionTransformer, expected: int | None = None) -> int:
    actual = count_parameters(model)
    derived = sum(value for _, value in parameter_table(model.config))
    if actual != derived:
        raise AssertionError(f"parameter tree has {actual:,}, derivation has {derived:,}")
    if expected is not None and actual != expected:
        raise AssertionError(f"expected {expected:,} trainable parameters, found {actual:,}")
    if model.config.is_exact_default and actual != 10_000_000:
        raise AssertionError(f"frozen default must have 10,000,000 parameters, found {actual:,}")
    return actual


### Canonical source: `losses.py`

In [ ]:
%%writefile src/jax_addition_transformer/losses.py
"""Answer-only autoregressive loss and metrics."""

from __future__ import annotations

import jax
import jax.numpy as jnp


def answer_mask(input_length: int = 15, answer_digits: int = 4) -> jax.Array:
    return jnp.arange(input_length) >= input_length - answer_digits


def masked_cross_entropy(
    logits: jax.Array, targets: jax.Array, mask: jax.Array | None = None
) -> tuple[jax.Array, jax.Array]:
    if logits.shape[:-1] != targets.shape:
        raise ValueError("logit and target shapes do not align")
    mask = answer_mask(targets.shape[1]) if mask is None else jnp.asarray(mask, dtype=bool)
    if mask.shape != (targets.shape[1],):
        raise ValueError("mask must have one value per target position")
    log_probs = jax.nn.log_softmax(logits.astype(jnp.float32), axis=-1)
    losses = -jnp.take_along_axis(log_probs, targets[..., None], axis=-1).squeeze(-1)
    denominator = targets.shape[0] * jnp.sum(mask)
    loss = jnp.sum(jnp.where(mask[None, :], losses, 0.0)) / denominator
    correct = jnp.argmax(logits, axis=-1) == targets
    accuracy = jnp.sum(jnp.where(mask[None, :], correct, False)) / denominator
    return loss, accuracy


### Canonical source: `optimizers.py`

In [ ]:
%%writefile src/jax_addition_transformer/optimizers.py
"""Optax schedules, optimizer variants, and path-based selective decay."""

from __future__ import annotations

from typing import Any

import jax
import optax

from .config import OptimizerConfig


def learning_rate_schedule(config: OptimizerConfig) -> optax.Schedule:
    return optax.warmup_cosine_decay_schedule(
        init_value=config.initial_learning_rate,
        peak_value=config.peak_learning_rate,
        warmup_steps=config.warmup_steps,
        decay_steps=config.total_steps,
        end_value=config.final_learning_rate,
    )


def _path_parts(path: tuple[Any, ...]) -> tuple[str, ...]:
    return tuple(str(getattr(entry, "key", getattr(entry, "idx", entry))) for entry in path)


def decay_mask(params):
    """Decay only 2-D `kernel` leaves owned by attention or FFN modules."""

    def eligible(path, leaf):
        parts = _path_parts(path)
        return bool(
            getattr(leaf, "ndim", 0) == 2
            and "kernel" in parts
            and ({"attention", "ffn"} & set(parts))
        )

    return jax.tree_util.tree_map_with_path(eligible, params)


def make_optimizer(config: OptimizerConfig, params):
    schedule = learning_rate_schedule(config)
    transformations: list[optax.GradientTransformation] = [
        optax.clip_by_global_norm(config.clip_global_norm)
    ]
    if config.name in {"adamw", "adam"}:
        transformations.append(
            optax.scale_by_adam(b1=config.beta1, b2=config.beta2, eps=config.epsilon)
        )
        if config.name == "adamw":
            transformations.append(
                optax.masked(optax.add_decayed_weights(config.weight_decay), decay_mask(params))
            )
    else:
        transformations.append(optax.trace(decay=config.momentum, nesterov=False))
    transformations.append(optax.scale_by_learning_rate(schedule))
    return optax.chain(*transformations), schedule


### Canonical source: `generation.py`

In [ ]:
%%writefile src/jax_addition_transformer/generation.py
"""Fixed-buffer compiled greedy autoregressive generation."""

from __future__ import annotations

import jax
import jax.numpy as jnp
import numpy as np

from .config import TaskConfig
from .data import operands_to_pair_ids, tokenize_pairs
from .tokenizer import decode_internal_answer, format_prompt, parse_expression, validate_operand

DEFAULT_TASK = TaskConfig()


def greedy_generate(
    model, prompts: jax.Array, answer_digits: int = 4
) -> tuple[jax.Array, jax.Array]:
    """Fill a fixed buffer. Future zeros are ordinary `0` tokens, not padding."""
    prompt_length = prompts.shape[1]
    maximum_length = prompt_length + answer_digits - 1
    buffer = jnp.zeros((prompts.shape[0], maximum_length), dtype=jnp.int32)
    buffer = buffer.at[:, :prompt_length].set(prompts)

    def body(current, offset):
        logits = model(current)
        token = jnp.argmax(logits[:, prompt_length + offset - 1, :], axis=-1).astype(jnp.int32)
        current = jax.lax.cond(
            offset < answer_digits - 1,
            lambda value: value.at[:, prompt_length + offset].set(token),
            lambda value: value,
            current,
        )
        return current, token

    _, generated_steps = jax.lax.scan(body, buffer, jnp.arange(answer_digits))
    generated = jnp.swapaxes(generated_steps, 0, 1)
    return generated, jnp.all(generated < 10, axis=-1)


def predict_batch(model_state, a_array, b_array, task: TaskConfig = DEFAULT_TASK):
    a, b = np.asarray(a_array), np.asarray(b_array)
    if a.shape != b.shape or a.ndim != 1:
        raise ValueError("a_array and b_array must be equally sized vectors")
    for value in np.concatenate((a, b)):
        validate_operand(value, task.max_digits)
    pair_ids = operands_to_pair_ids(a, b, 10**task.max_digits)
    prompt_length = 2 * task.max_digits + 6
    prompts = tokenize_pairs(pair_ids, task.max_digits)[:, :prompt_length]
    return greedy_generate(model_state, jnp.asarray(prompts), task.answer_digits)


def predict_pair(model_state, a: int, b: int, task: TaskConfig = DEFAULT_TASK) -> str:
    generated, valid = predict_batch(model_state, [a], [b], task)
    if not bool(valid[0]):
        return "<invalid>"
    return decode_internal_answer(np.asarray(generated[0]))


def ask(model_state, expression: str, task: TaskConfig = DEFAULT_TASK, debug: bool = False) -> str:
    a, b = parse_expression(expression, task.max_digits)
    prompt = format_prompt(a, b, task.max_digits)
    generated, valid = predict_batch(model_state, [a], [b], task)
    internal = np.asarray(generated[0])
    result = decode_internal_answer(internal) if bool(valid[0]) else "<invalid>"
    if debug:
        return f"prompt={prompt!r} internal={''.join(map(str, internal))} result={result}"
    return result


### Canonical source: `evaluation.py`

In [ ]:
%%writefile src/jax_addition_transformer/evaluation.py
"""Teacher-forced and compiled greedy evaluation with exact slice accounting."""

from __future__ import annotations

import jax.numpy as jnp
import numpy as np
from flax import nnx

from .data import carry_pattern, operand_lengths, pair_ids_to_operands, tokenize_pairs
from .generation import greedy_generate
from .losses import masked_cross_entropy


def evaluate_ids(
    model, pair_ids: np.ndarray, batch_size: int = 4000, max_digits: int = 3
) -> tuple[dict, list[dict]]:
    if len(pair_ids) % batch_size:
        raise ValueError("evaluation size must be divisible by batch_size")
    correct_all, failures, predictions = [], [], []
    token_correct, token_total, loss_sum = 0, 0, 0.0
    group_counts: dict[str, dict[str, list[int]]] = {
        "operand_length": {},
        "carry_pattern": {},
        "answer_length": {},
    }
    forward = nnx.jit(lambda current_model, tokens: current_model(tokens))
    generate = nnx.jit(
        lambda current_model, prompts: greedy_generate(current_model, prompts, max_digits + 1)
    )

    def record_group(group: str, key: str, is_correct: bool) -> None:
        counts = group_counts[group].setdefault(key, [0, 0])
        counts[0] += int(is_correct)
        counts[1] += 1

    for start in range(0, len(pair_ids), batch_size):
        ids = pair_ids[start : start + batch_size]
        full = tokenize_pairs(ids, max_digits)
        inputs, targets = jnp.asarray(full[:, :-1]), jnp.asarray(full[:, 1:])
        logits = forward(model, inputs)
        loss, accuracy = masked_cross_entropy(logits, targets)
        loss_sum += float(loss) * batch_size
        token_correct += float(accuracy) * batch_size * (max_digits + 1)
        token_total += batch_size * (max_digits + 1)
        prompt_length = 2 * max_digits + 6
        generated, valid = generate(model, inputs[:, :prompt_length])
        generated, valid = np.asarray(generated), np.asarray(valid)
        expected = full[:, -max_digits - 1 :]
        correct = valid & np.all(generated == expected, axis=1)
        correct_all.extend(correct.tolist())
        predictions.extend(
            [tuple(map(int, row)) if ok else None for row, ok in zip(generated, valid, strict=True)]
        )
        a, b = pair_ids_to_operands(ids, 10**max_digits)
        lengths_a, lengths_b = operand_lengths(a), operand_lengths(b)
        patterns = carry_pattern(a, b, max_digits)
        sums = a + b
        answer_lengths = np.where(sums >= 10**max_digits, max_digits + 1, operand_lengths(sums))
        for i, solved in enumerate(correct):
            record_group("operand_length", f"{lengths_a[i]}x{lengths_b[i]}", bool(solved))
            record_group("carry_pattern", f"{patterns[i]:0{max_digits}b}", bool(solved))
            record_group("answer_length", str(answer_lengths[i]), bool(solved))
        for i in np.flatnonzero(~correct):
            pred = (
                ("".join(map(str, generated[i][::-1])).lstrip("0") or "0")
                if valid[i]
                else "<invalid>"
            )
            failures.append(
                {
                    "pair_id": int(ids[i]),
                    "a": int(a[i]),
                    "b": int(b[i]),
                    "target": str(int(sums[i])),
                    "prediction": pred,
                    "internal_generated_digits": "".join(map(str, generated[i])),
                    "valid_digit_sequence": bool(valid[i]),
                    "operand_length_a": int(lengths_a[i]),
                    "operand_length_b": int(lengths_b[i]),
                    "carry_pattern": f"{patterns[i]:0{max_digits}b}",
                }
            )
    slices = {
        group: {
            key: {"correct": values[0], "total": values[1], "accuracy": values[0] / values[1]}
            for key, values in sorted(rows.items())
        }
        for group, rows in group_counts.items()
    }
    position = {int(pair_id): index for index, pair_id in enumerate(pair_ids)}
    comparable, consistent, base = 0, 0, 10**max_digits
    for index, pair_id in enumerate(pair_ids):
        a, b = divmod(int(pair_id), base)
        swapped = base * b + a
        if swapped in position:
            comparable += 1
            consistent += predictions[index] == predictions[position[swapped]]
    metrics = {
        "teacher_forced_loss": loss_sum / len(pair_ids),
        "teacher_forced_token_accuracy": token_correct / token_total,
        "greedy_exact_match": sum(correct_all) / len(correct_all),
        "failure_count": len(failures),
        "invalid_token_count": sum(not row["valid_digit_sequence"] for row in failures),
        "commutativity_consistency": consistent / comparable if comparable else None,
        "commutativity_comparable_pairs": comparable,
        "slices": slices,
    }
    return metrics, failures


### Canonical source: `checkpointing.py`

In [ ]:
%%writefile src/jax_addition_transformer/checkpointing.py
"""Orbax PyTree checkpoints and reproducibility metadata."""

from __future__ import annotations

import importlib.metadata
import json
import platform
import subprocess
from datetime import datetime, UTC
from pathlib import Path

import jax
import numpy as np
import orbax.checkpoint as ocp


def capture_environment(config) -> dict:
    def version(name: str) -> str | None:
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None

    try:
        commit = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
        ).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        commit = None
    return {
        "timestamp": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "operating_system": platform.platform(),
        "jax": jax.__version__,
        "jaxlib": version("jaxlib"),
        "flax": version("flax"),
        "optax": version("optax"),
        "orbax": version("orbax-checkpoint"),
        "numpy": np.__version__,
        "devices": [str(d) for d in jax.devices()],
        "device_count": jax.device_count(),
        "backend": jax.default_backend(),
        "compute_dtype": config.model.compute_dtype,
        "git_commit": commit,
        "configuration_hash": config.fingerprint,
    }


def save_checkpoint(path: str | Path, params, optimizer_state, metadata: dict) -> None:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    with ocp.Checkpointer(ocp.StandardCheckpointHandler(use_ocdbt=False)) as checkpointer:
        checkpointer.save(
            path / "state",
            args=ocp.args.StandardSave({"params": params, "optimizer_state": optimizer_state}),
            force=True,
        )
    (path / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")


def restore_checkpoint(
    path: str | Path, params, optimizer_state, expected_fingerprint: str | None = None
):
    path = Path(path)
    metadata = json.loads((path / "metadata.json").read_text())
    if expected_fingerprint is not None and metadata["config_fingerprint"] != expected_fingerprint:
        raise ValueError("checkpoint configuration fingerprint is incompatible")
    with ocp.Checkpointer(ocp.StandardCheckpointHandler(use_ocdbt=False)) as checkpointer:
        restored = checkpointer.restore(
            path / "state",
            args=ocp.args.StandardRestore({"params": params, "optimizer_state": optimizer_state}),
        )
    return restored["params"], restored["optimizer_state"], metadata


### Canonical source: `reporting.py`

In [ ]:
%%writefile src/jax_addition_transformer/reporting.py
"""Reports generated only from existing run artifacts."""

from __future__ import annotations

import json
from pathlib import Path


def generate_report(run_dir: str | Path) -> Path:
    run_dir = Path(run_dir)
    config = json.loads((run_dir / "config.json").read_text())
    environment = json.loads((run_dir / "environment.json").read_text())
    split = json.loads((run_dir / "split_metadata.json").read_text())
    summary = json.loads((run_dir / "summary.json").read_text())
    history = [json.loads(line) for line in (run_dir / "history.jsonl").read_text().splitlines()]
    validation_history = [record["validation"] for record in history if "validation" in record]
    final_validation = validation_history[-1] if validation_history else {}
    evaluations = summary.get("evaluations", {})

    def metric(split_name, key):
        return evaluations.get(split_name, {}).get(key, "not evaluated")

    best = summary.get("best") or {}
    lines = [
        "# Experiment report",
        "",
        "This report is generated only from recorded run artifacts.",
        "",
        "## Configuration",
        "",
        "```json",
        json.dumps(config, indent=2),
        "```",
        "",
        "## Recorded results",
        "",
        f"- Parameter count: {summary['parameter_count']:,}",
        f"- Split fingerprint: `{split['fingerprint']}`",
        f"- Compilation time: {summary.get('compilation_seconds', 'not evaluated')}",
        f"- Steady-state training time: {summary.get('steady_state_training_seconds', 'not evaluated')}",
        f"- Total training time: {summary.get('total_training_seconds', 'not evaluated')}",
        f"- Best step: {best.get('step', 'not evaluated')}",
        f"- Final train loss: {summary.get('final_train_loss', 'not evaluated')}",
        f"- Final validation loss: {final_validation.get('teacher_forced_loss', 'not evaluated')}",
        f"- Best validation loss: {best.get('loss', 'not evaluated')}",
        f"- Final answer-token accuracy: {summary.get('final_answer_token_accuracy', 'not evaluated')}",
        f"- Final validation token accuracy: {final_validation.get('teacher_forced_token_accuracy', 'not evaluated')}",
        f"- Validation exact match: {metric('validation', 'greedy_exact_match')}",
        f"- Unseen-test exact match: {metric('test', 'greedy_exact_match')}",
        f"- Exhaustive exact match: {metric('exhaustive', 'greedy_exact_match')}",
        f"- Failure count: {metric('exhaustive', 'failure_count')}",
        "",
        "## Carry-pattern and operand-length tables",
        "",
        "```json",
        json.dumps(evaluations.get("exhaustive", {}).get("slices", "not evaluated"), indent=2),
        "```",
        "",
        "## Environment",
        "",
        "```json",
        json.dumps(environment, indent=2),
        "```",
        "",
        "## Sample predictions",
        "",
        (run_dir / "sample_predictions.txt").read_text(),
        "",
        "Checkpoints: `checkpoints/best` (when validation has run), `checkpoints/latest` (resume state)",
        "",
    ]
    path = run_dir / "report.md"
    path.write_text("\n".join(lines))
    return path


### Canonical source: `training.py`

In [ ]:
%%writefile src/jax_addition_transformer/training.py
"""Pure-state compiled step and explicit Python training loop."""

from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Any

import jax
import jax.numpy as jnp
import optax
from flax import nnx

from .checkpointing import capture_environment, restore_checkpoint, save_checkpoint
from .config import ExperimentConfig
from .data import create_split
from .losses import masked_cross_entropy
from .model import AdditionTransformer, assert_parameter_count
from .optimizers import make_optimizer
from .sampling import HybridSampler
from .evaluation import evaluate_ids


def tree_all_finite(tree) -> jax.Array:
    return jnp.all(jnp.stack([jnp.all(jnp.isfinite(x)) for x in jax.tree.leaves(tree)]))


def _raise_nonfinite() -> None:
    raise FloatingPointError(
        "non-finite loss, gradients, or updated parameters in compiled training step"
    )


def create_train_step(graphdef, optimizer):
    """Close over static graph/optimizer once; pass only array PyTrees to JIT."""

    @jax.jit
    def train_step(params, optimizer_state, inputs, targets):
        def objective(candidate):
            model = nnx.merge(graphdef, candidate)
            loss, accuracy = masked_cross_entropy(model(inputs), targets)
            return loss, accuracy

        (loss, accuracy), gradients = jax.value_and_grad(objective, has_aux=True)(params)
        gradient_norm = optax.global_norm(gradients)
        finite = jnp.isfinite(loss) & tree_all_finite(gradients)
        updates, optimizer_state = optimizer.update(gradients, optimizer_state, params)
        params = optax.apply_updates(params, updates)
        finite = finite & tree_all_finite(params)

        def fail(_):
            jax.debug.callback(_raise_nonfinite, ordered=True)
            return jnp.int32(0)

        jax.lax.cond(finite, lambda _: jnp.int32(0), fail, operand=None)
        return (
            params,
            optimizer_state,
            {
                "loss": loss,
                "answer_token_accuracy": accuracy,
                "gradient_global_norm": gradient_norm,
                "finite": finite,
            },
        )

    return train_step


def train(
    config: ExperimentConfig,
    run_dir: str | Path,
    smoke_steps: int | None = None,
    resume: bool = False,
) -> dict[str, Any]:
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "checkpoints").mkdir(exist_ok=True)
    config_path = run_dir / "config.json"
    latest = run_dir / "checkpoints" / "latest"
    if resume:
        if not latest.exists() or not config_path.exists():
            raise FileNotFoundError(f"cannot resume: {latest} or {config_path} does not exist")
        if ExperimentConfig.load(config_path).fingerprint != config.fingerprint:
            raise ValueError("resume configuration is incompatible with the existing run")
    elif latest.exists():
        raise FileExistsError(
            f"run already contains a latest checkpoint; pass resume=True: {latest}"
        )
    else:
        config.save(config_path)
    split = create_split(
        config.training.seed,
        config.training.train_pairs,
        config.training.validation_pairs,
        10**config.task.max_digits,
    )
    split_metadata = {
        "seed": split.seed,
        "train_size": len(split.train),
        "validation_size": len(split.validation),
        "test_size": len(split.test),
        "domain_size": len(split.train) + len(split.validation) + len(split.test),
        "pair_id_formula": f"{10**config.task.max_digits} * a + b",
        "stratification": "operand length of a, operand length of b, carry bits ordered units-tens-hundreds",
        "allocation": "deterministic largest remainder within strata",
        "fingerprint_algorithm": "SHA-256 over named little-endian int64 split arrays",
        "fingerprint": split.fingerprint,
    }
    (run_dir / "split_metadata.json").write_text(json.dumps(split_metadata, indent=2) + "\n")
    model = AdditionTransformer(config.model, rngs=nnx.Rngs(params=config.training.seed))
    parameter_count = assert_parameter_count(
        model, 10_000_000 if config.model.is_exact_default else None
    )
    graphdef, params = nnx.split(model, nnx.Param)
    optimizer, schedule = make_optimizer(config.optimizer, params)
    optimizer_state = optimizer.init(params)
    step_fn = create_train_step(graphdef, optimizer)
    sampler = HybridSampler.create(
        split.train, config.training.batch_size, config.training.seed, 10**config.task.max_digits
    )
    environment = capture_environment(config)
    (run_dir / "environment.json").write_text(json.dumps(environment, indent=2) + "\n")
    total_steps = smoke_steps if smoke_steps is not None else config.training.max_steps
    history_path = run_dir / "history.jsonl"
    start_step, best = 0, None
    if resume:
        params, optimizer_state, metadata = restore_checkpoint(
            latest, params, optimizer_state, config.fingerprint
        )
        start_step = int(metadata["step"])
        sampler.restore_state(metadata["sampler_state"])
        best = metadata["best"]
        history = metadata["history"]
        history_path.write_text("".join(json.dumps(record) + "\n" for record in history))
    else:
        history = []
        history_path.write_text("")
    started = time.perf_counter()
    compile_seconds = None
    step_durations = []
    for step in range(start_step + 1, total_steps + 1):
        inputs, targets = sampler.sample_batch(config.task.max_digits)
        before = time.perf_counter()
        params, optimizer_state, metrics = step_fn(
            params, optimizer_state, jnp.asarray(inputs), jnp.asarray(targets)
        )
        jax.block_until_ready(metrics["loss"])
        duration = time.perf_counter() - before
        step_durations.append(duration)
        if compile_seconds is None:
            compile_seconds = duration
        checkpoint_interval = 1 if smoke_steps is not None else config.training.checkpoint_interval
        checkpoint_due = step % checkpoint_interval == 0
        validation_due = smoke_steps is None and step % config.training.validation_interval == 0
        should_record = (
            step == start_step + 1
            or step == total_steps
            or checkpoint_due
            or validation_due
            or step % config.training.logging_interval == 0
        )
        if not should_record:
            continue
        host = {name: float(value) for name, value in jax.device_get(metrics).items()}
        if not bool(host["finite"]):
            raise FloatingPointError(f"non-finite loss, gradients, or parameters at step {step}")
        record = {
            "step": step,
            **host,
            "learning_rate": float(schedule(step - 1)),
            "step_seconds": duration,
            "examples_per_second": config.training.batch_size / duration,
            "answer_tokens_per_second": config.training.batch_size
            * config.task.answer_digits
            / duration,
        }
        save_as_best = False
        if validation_due:
            validation_model = nnx.merge(graphdef, params)
            validation, _ = evaluate_ids(
                validation_model,
                split.validation,
                config.training.evaluation_batch_size,
                config.task.max_digits,
            )
            candidate = {
                "step": step,
                "greedy_exact_match": validation["greedy_exact_match"],
                "loss": validation["teacher_forced_loss"],
            }
            record["validation"] = validation
            save_as_best = best is None or (
                candidate["greedy_exact_match"] > best["greedy_exact_match"]
                or (
                    candidate["greedy_exact_match"] == best["greedy_exact_match"]
                    and candidate["loss"] < best["loss"]
                )
            )
            if save_as_best:
                best = candidate
        history.append(record)
        with history_path.open("a") as handle:
            handle.write(json.dumps(record) + "\n")
        if step == start_step + 1 or step % config.training.logging_interval == 0:
            print(json.dumps(record))
        if save_as_best:
            save_checkpoint(
                run_dir / "checkpoints" / "best",
                params,
                optimizer_state,
                {
                    "step": step,
                    "sampler_state": sampler.state,
                    "best": best,
                    "history": history,
                    "config_fingerprint": config.fingerprint,
                },
            )
        if checkpoint_due or step == total_steps:
            save_checkpoint(
                run_dir / "checkpoints" / "latest",
                params,
                optimizer_state,
                {
                    "step": step,
                    "sampler_state": sampler.state,
                    "best": best,
                    "history": history,
                    "config_fingerprint": config.fingerprint,
                },
            )
    previous_summary = {}
    if resume and (run_dir / "summary.json").exists():
        previous_summary = json.loads((run_dir / "summary.json").read_text())
    segment_seconds = time.perf_counter() - started
    steady_state_seconds = sum(step_durations[1:]) if len(step_durations) > 1 else 0.0
    summary = {
        "parameter_count": parameter_count,
        "steps": total_steps,
        "compilation_seconds": previous_summary.get("compilation_seconds", compile_seconds),
        "resume_compilation_seconds": compile_seconds if resume else None,
        "steady_state_training_seconds": previous_summary.get("steady_state_training_seconds", 0.0)
        + steady_state_seconds,
        "training_segment_seconds": segment_seconds,
        "total_training_seconds": previous_summary.get("total_training_seconds", 0.0)
        + segment_seconds,
        "best": best,
        "final_train_loss": history[-1]["loss"] if history else "not evaluated",
        "final_answer_token_accuracy": history[-1]["answer_token_accuracy"]
        if history
        else "not evaluated",
        "evaluations": previous_summary.get("evaluations", {}),
    }
    (run_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    for name, content in (
        (
            "failures.csv",
            "pair_id,a,b,target,prediction,internal_generated_digits,valid_digit_sequence,operand_length_a,operand_length_b,carry_pattern,split\n",
        ),
        ("sample_predictions.txt", "No predictions evaluated yet.\n"),
    ):
        (run_dir / name).write_text(content)
    return summary


### Canonical source: `cli.py`

In [ ]:
%%writefile src/jax_addition_transformer/cli.py
"""Argparse console entry points."""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from flax import nnx

from .checkpointing import restore_checkpoint
from .config import ExperimentConfig
from .data import create_split
from .evaluation import evaluate_ids
from .generation import ask, predict_pair
from .model import AdditionTransformer, assert_parameter_count, parameter_table
from .optimizers import make_optimizer
from .reporting import generate_report
from .training import train


def _config_argument(parser: argparse.ArgumentParser) -> None:
    parser.add_argument("--config", default="configs/exact_10m_t4.json")


def inspect_main() -> None:
    parser = argparse.ArgumentParser(description="Inspect an arithmetic-transformer configuration")
    _config_argument(parser)
    args = parser.parse_args()
    config = ExperimentConfig.load(args.config)
    model = AdditionTransformer(config.model, rngs=nnx.Rngs(params=config.training.seed))
    total = assert_parameter_count(model, 10_000_000 if config.model.is_exact_default else None)
    print("Component                         Parameters")
    print("-------------------------------- ----------")
    for name, count in parameter_table(config.model):
        print(f"{name:32} {count:10,d}")
    print("-------------------------------- ----------")
    print(f"Total trainable parameters       {total:10,d}")


def train_main() -> None:
    parser = argparse.ArgumentParser(description="Train the arithmetic transformer")
    _config_argument(parser)
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--smoke-steps", type=int)
    parser.add_argument("--resume", action="store_true")
    args = parser.parse_args()
    summary = train(ExperimentConfig.load(args.config), args.run_dir, args.smoke_steps, args.resume)
    print(json.dumps(summary, indent=2))


def _restore_model(run_dir: Path, checkpoint: str = "best"):
    config = ExperimentConfig.load(run_dir / "config.json")
    model = AdditionTransformer(config.model, rngs=nnx.Rngs(params=config.training.seed))
    graphdef, params = nnx.split(model, nnx.Param)
    optimizer, _ = make_optimizer(config.optimizer, params)
    optimizer_state = optimizer.init(params)
    checkpoint_path = run_dir / "checkpoints" / checkpoint
    if checkpoint == "best" and not checkpoint_path.exists():
        checkpoint_path = run_dir / "checkpoints" / "latest"
    params, _, _ = restore_checkpoint(checkpoint_path, params, optimizer_state, config.fingerprint)
    return nnx.merge(graphdef, params), config


def eval_main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate a saved run")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--split", choices=["validation", "test"], default="test")
    parser.add_argument("--exhaustive", action="store_true")
    parser.add_argument("--checkpoint", choices=["best", "latest"], default="best")
    args = parser.parse_args()
    run_dir = Path(args.run_dir)
    model, config = _restore_model(run_dir, args.checkpoint)
    split = create_split(
        config.training.seed,
        config.training.train_pairs,
        config.training.validation_pairs,
        10**config.task.max_digits,
    )
    ids = (
        np.arange((10**config.task.max_digits) ** 2)
        if args.exhaustive
        else getattr(split, args.split)
    )
    metrics, failures = evaluate_ids(
        model, ids, config.training.evaluation_batch_size, config.task.max_digits
    )
    fieldnames = [
        "pair_id",
        "a",
        "b",
        "target",
        "prediction",
        "internal_generated_digits",
        "valid_digit_sequence",
        "operand_length_a",
        "operand_length_b",
        "carry_pattern",
        "split",
    ]
    with (run_dir / "failures.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        if args.exhaustive:
            split_labels = np.empty((len(ids),), dtype="<U10")
            split_labels[split.train] = "train"
            split_labels[split.validation] = "validation"
            split_labels[split.test] = "test"
        for row in failures:
            label = split_labels[row["pair_id"]] if args.exhaustive else args.split
            writer.writerow({**row, "split": label})
    summary_path = run_dir / "summary.json"
    summary = json.loads(summary_path.read_text())
    key = "exhaustive" if args.exhaustive else args.split
    summary.setdefault("evaluations", {})[key] = metrics
    summary_path.write_text(json.dumps(summary, indent=2) + "\n")
    examples = [
        (0, 0),
        (1, 9),
        (9, 1),
        (9, 991),
        (99, 1),
        (199, 801),
        (499, 501),
        (500, 500),
        (909, 91),
        (999, 1),
        (999, 999),
    ]
    examples = [
        (a, b)
        for a, b in examples
        if a <= config.task.maximum_operand and b <= config.task.maximum_operand
    ]
    (run_dir / "sample_predictions.txt").write_text(
        "\n".join(
            f"{a:0{config.task.max_digits}d} + {b:0{config.task.max_digits}d} = {predict_pair(model, a, b, config.task)}"
            for a, b in examples
        )
        + "\n"
    )
    print(json.dumps(metrics, indent=2))


def chat_main() -> None:
    parser = argparse.ArgumentParser(description="Interactive model-only arithmetic inference")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--checkpoint", choices=["best", "latest"], default="best")
    args = parser.parse_args()
    model, config = _restore_model(Path(args.run_dir), args.checkpoint)
    print(f"Enter additions using operands 0-{config.task.maximum_operand}; Ctrl-D exits.")
    while True:
        try:
            expression = input("> ")
        except EOFError:
            break
        try:
            print(ask(model, expression, config.task))
        except (TypeError, ValueError) as exc:
            print(f"error: {exc}")


def report_main() -> None:
    parser = argparse.ArgumentParser(description="Generate a report from real artifacts")
    parser.add_argument("--run-dir", required=True)
    args = parser.parse_args()
    print(generate_report(args.run_dir))


### Canonical source: `__init__.py`

In [ ]:
%%writefile src/jax_addition_transformer/__init__.py
"""From-scratch arithmetic transformer built with JAX and Flax NNX."""

from .config import ExperimentConfig, ModelConfig, OptimizerConfig, TaskConfig, TrainingConfig
from .model import AdditionTransformer

__all__ = [
    "AdditionTransformer",
    "ExperimentConfig",
    "ModelConfig",
    "OptimizerConfig",
    "TaskConfig",
    "TrainingConfig",
]
__version__ = "0.1.0"


In [ ]:
import sys
sys.path.insert(0, 'src')
from jax_addition_transformer.config import ExperimentConfig
from jax_addition_transformer.model import AdditionTransformer, assert_parameter_count, parameter_table
from flax import nnx
config = ExperimentConfig.load('configs/exact_10m_t4.json') if __import__('pathlib').Path('configs/exact_10m_t4.json').exists() else ExperimentConfig()
model = AdditionTransformer(config.model, rngs=nnx.Rngs(params=42))
print(parameter_table(config.model)); print('parameters:', assert_parameter_count(model, 10_000_000))

## 4. Architecture, tensor shapes, and causal mask

In [ ]:
architecture=[['model','decoder-only causal'],['layers',config.model.n_layers],['width',config.model.d_model],['heads',config.model.n_heads],['head dimension',config.model.head_dim],['FFN',config.model.d_ff],['normalization',config.model.norm_type],['positions',config.model.position_type],['weight tying',config.model.tie_embeddings]]
fig,ax=plt.subplots(figsize=(7,3)); ax.axis('off'); ax.table(cellText=architecture,colLabels=['decision','default'],loc='center'); ax.set_title('Frozen default architecture decisions'); plt.show()

In [ ]:
from jax_addition_transformer.attention import causal_mask
mask=causal_mask(15); assert bool(mask[14,0]) and not bool(mask[0,14])
fig, ax = plt.subplots(); ax.imshow(mask, cmap='Blues'); ax.set(title='Causal visibility mask', xlabel='key position', ylabel='query position'); plt.show()

In [ ]:
names, values = zip(*parameter_table(config.model)); fig, ax=plt.subplots(figsize=(8,3)); ax.barh(names,values); ax.set(title='Trainable parameter breakdown',xlabel='parameters',ylabel='component'); plt.show()

A Pre-LayerNorm block follows `x → norm → causal attention → +x → norm → FFN → +`. Default Q/K/V are `[batch, 15, 5, 64]`; scores and probabilities are `[batch, 5, 15, 15]`.

## 5. Dataset, split, and hybrid sampler

In [ ]:
from jax_addition_transformer.data import create_split, pair_ids_to_operands, operand_lengths, carry_pattern
from jax_addition_transformer.sampling import HybridSampler
split=create_split(); a,b=pair_ids_to_operands(split.train); print(split.fingerprint, len(split.train),len(split.validation),len(split.test))
fig,ax=plt.subplots(); ax.hist2d(operand_lengths(a),operand_lengths(b),bins=3); ax.set(title='Training pairs by operand length',xlabel='length(a)',ylabel='length(b)'); plt.show()

In [ ]:
sampler=HybridSampler.create(split.train,2048,42); natural=carry_pattern(a,b); ids=sampler.sample_ids(); sa,sb=pair_ids_to_operands(ids); sampled=carry_pattern(sa,sb)
fig,ax=plt.subplots(); x=range(8); ax.bar([i-.2 for i in x],[(natural==i).mean() for i in x],.4,label='natural'); ax.bar([i+.2 for i in x],[(sampled==i).mean() for i in x],.4,label='hybrid'); ax.set(title='Natural versus hybrid carry distribution',xlabel='carry code (units bit is leftmost)',ylabel='fraction'); ax.legend(); plt.show()

## 6. Optimizer and learning-rate schedule

In [ ]:
from jax_addition_transformer.optimizers import learning_rate_schedule
schedule=learning_rate_schedule(config.optimizer); steps=range(config.optimizer.total_steps); fig,ax=plt.subplots(); ax.plot(steps,[float(schedule(s)) for s in steps]); ax.set(title='Warmup-cosine learning-rate schedule',xlabel='step',ylabel='learning rate'); plt.show()

## 7. Training, checkpointing, and measured curves

In [ ]:
from jax_addition_transformer.training import train
RUN_DIR='runs/colab-default'
if FULL_RUN:
    require_gpu_for_full_run(); resume_run=(Path(RUN_DIR)/'checkpoints'/'latest').exists(); summary=train(config,RUN_DIR,resume=resume_run)
else:
    print('Training skipped: select a T4 GPU for the full run.')

In [ ]:
from jax_addition_transformer.checkpointing import restore_checkpoint
from jax_addition_transformer.optimizers import make_optimizer
if FULL_RUN:
 graphdef, empty_params = nnx.split(model, nnx.Param)
 optimizer, _ = make_optimizer(config.optimizer, empty_params); empty_optimizer_state = optimizer.init(empty_params)
 chosen = Path(RUN_DIR)/'checkpoints'/'best'
 if not chosen.exists(): chosen = Path(RUN_DIR)/'checkpoints'/'latest'
 restored_params, _, checkpoint_metadata = restore_checkpoint(chosen, empty_params, empty_optimizer_state, config.fingerprint)
 model = nnx.merge(graphdef, restored_params)
 print('Restored trained checkpoint at step', checkpoint_metadata['step'])

In [ ]:
import json, pathlib
p=pathlib.Path(RUN_DIR)/'history.jsonl'
if p.exists():
 h=[json.loads(x) for x in p.read_text().splitlines()]; fields=[('loss','Training loss'),('answer_token_accuracy','Answer-token accuracy'),('gradient_global_norm','Gradient norm'),('examples_per_second','Examples per second')]
 fig,axs=plt.subplots(2,2,figsize=(10,7))
 for ax,(field,title) in zip(axs.flat,fields): ax.plot([r['step'] for r in h],[r[field] for r in h]); ax.set(title=title,xlabel='step',ylabel=field)
 plt.tight_layout(); plt.show()
else: print('No history exists; no curves are invented.')

In [ ]:
if p.exists():
 vh=[r for r in h if 'validation' in r]
 fig,axs=plt.subplots(1,2,figsize=(10,3))
 if vh:
  axs[0].plot([r['step'] for r in vh],[r['validation']['teacher_forced_loss'] for r in vh],marker='o'); axs[1].plot([r['step'] for r in vh],[r['validation']['greedy_exact_match'] for r in vh],marker='o')
 for ax,title,ylabel in zip(axs,['Validation loss','Validation greedy exact match'],['loss','exact-match accuracy']): ax.set(title=title,xlabel='step',ylabel=ylabel)
 plt.tight_layout(); plt.show()
else: print('No measured validation history exists.')

## 8. Greedy validation, unseen test, exhaustive evaluation, and slices

In [ ]:
from jax_addition_transformer.evaluation import evaluate_ids
if FULL_RUN:
 validation_metrics, _ = evaluate_ids(model,split.validation,config.training.evaluation_batch_size)
 test_metrics, _ = evaluate_ids(model,split.test,config.training.evaluation_batch_size)
 exhaustive_metrics, failures = evaluate_ids(model,__import__('numpy').arange(1_000_000),config.training.evaluation_batch_size)
 print('validation',validation_metrics); print('unseen test',test_metrics); print('exhaustive',exhaustive_metrics)
else: print('Evaluation follows a real trained checkpoint; skipped without a full run.')

In [ ]:
if FULL_RUN:
 fig,axs=plt.subplots(1,2,figsize=(12,4))
 carry=exhaustive_metrics['slices']['carry_pattern']; lengths=exhaustive_metrics['slices']['operand_length']
 axs[0].bar(carry,[carry[k]['accuracy'] for k in carry]); axs[0].set(title='Accuracy by carry pattern',xlabel='units–tens–hundreds carries',ylabel='exact-match accuracy')
 axs[1].bar(lengths,[lengths[k]['accuracy'] for k in lengths]); axs[1].set(title='Accuracy by operand-length pair',xlabel='length(a) × length(b)',ylabel='exact-match accuracy')
 plt.tight_layout(); plt.show()

In [ ]:
if FULL_RUN and failures:
 invalid=sum(not row['valid_digit_sequence'] for row in failures); incorrect=len(failures)-invalid
 fig,ax=plt.subplots(); ax.bar(['incorrect digits','invalid token'],[incorrect,invalid]); ax.set(title='Failure summary',xlabel='failure type',ylabel='count'); plt.show()
elif FULL_RUN: print('No failures in this evaluated split.')

The CLI evaluation writes every failure to CSV and separately supports the unseen test and exhaustive million-pair domain. Slice plots and failure summaries must be drawn from those generated artifacts; absent results are displayed as not evaluated.

In [ ]:
if FULL_RUN:
 import csv, numpy as np
 fields=['pair_id','a','b','target','prediction','internal_generated_digits','valid_digit_sequence','operand_length_a','operand_length_b','carry_pattern','split']; split_labels=np.empty(1_000_000,dtype='<U10'); split_labels[split.train]='train'; split_labels[split.validation]='validation'; split_labels[split.test]='test'
 with open(pathlib.Path(RUN_DIR)/'failures.csv','w',newline='') as handle:
  writer=csv.DictWriter(handle,fieldnames=fields); writer.writeheader()
  for row in failures: writer.writerow({**row,'split':split_labels[row['pair_id']]})
 summary_path=pathlib.Path(RUN_DIR)/'summary.json'; summary=json.loads(summary_path.read_text()); summary['evaluations']={'validation':validation_metrics,'test':test_metrics,'exhaustive':exhaustive_metrics}; summary_path.write_text(json.dumps(summary,indent=2)+'\n')
 print('Wrote',len(failures),'failures without inventing results.')

## 9. Attention inspection, restore, and interactive inference

In [ ]:
from jax_addition_transformer.tokenizer import encode,format_training_example
import jax.numpy as jnp
if FULL_RUN:
 failed_ids={row['pair_id'] for row in failures}; solved_id=next((i for i in range(1_000_000) if i not in failed_ids),None)
 if solved_id is not None:
  solved_a,solved_b=divmod(solved_id,1000); logits,maps=model(jnp.asarray([encode(format_training_example(solved_a,solved_b)[:-1])]),return_attention=True)
  fig,axs=plt.subplots(1,config.model.n_heads,figsize=(15,3))
  for i,ax in enumerate(axs): ax.imshow(maps[-1][0,i]); ax.set(title=f'head {i}',xlabel='key',ylabel='query')
  fig.suptitle(f'Final-block attention for solved expression {solved_a:03d} + {solved_b:03d}'); plt.tight_layout(); plt.show()
 else: print('No solved expression exists, so no solved-example attention plot is claimed.')

In [ ]:
from jax_addition_transformer.generation import ask,predict_pair
examples=[(0,0),(1,9),(9,1),(9,991),(99,1),(199,801),(499,501),(500,500),(909,91),(999,1),(999,999)]
if FULL_RUN:
 sample_text='\n'.join(f'{a:03d} + {b:03d} = {predict_pair(model,a,b)}' for a,b in examples); print(sample_text); (pathlib.Path(RUN_DIR)/'sample_predictions.txt').write_text(sample_text+'\n')
else: print('Mandatory model examples require a trained checkpoint.')

In [ ]:
if FULL_RUN:
 second_params,_,_=restore_checkpoint(chosen,empty_params,empty_optimizer_state,config.fingerprint); second_model=nnx.merge(graphdef,second_params)
 before=predict_pair(model,123,456); after=predict_pair(second_model,123,456); assert before==after; print('Checkpoint round trip preserved prediction:',after)

In [ ]:
from jax_addition_transformer.reporting import generate_report
if FULL_RUN: print('Generated report:',generate_report(RUN_DIR))

In [ ]:
if FULL_RUN:
 print(ask(model,input('addition> '),debug=True))
else: print('Interactive inference requires a trained checkpoint.')

## 10. Configuration knobs and artifacts

`ModelConfig` exposes normalization, positions, FFN, attention grouping, dimensions, bias, tying, precision, epsilon, RoPE base, and initialization scale. A completed run writes configuration, environment, split metadata, history, summary, report, failures, samples, and best/latest checkpoints.